In [1]:
import os
import re
import xarray as xr
import numpy as np
from pathlib import Path

# =========================================================
# 1. 根目录
# =========================================================
root_dir = Path("/Users/zhangnan/日常文件/ai/每周下载和预测")

# =========================================================
# 2. 当前目录
# =========================================================
current_folder = "53 20260831"

dir_download = root_dir / current_folder / "数据" / "download" 
dir_out = root_dir / current_folder / "数据" / "data"
os.makedirs(dir_out, exist_ok=True)

# =========================================================
# 3. 合并规则
# =========================================================
merge_rules = {
    "Geopotential": 20,
    "SpecificHumidity": 20,
    "CloudFraction": 20,
    "Divergence": 20,
    "PotentialVorticity": 20,
    "2m_Temperature": 20,
    "MSLP": 20,
    "Totalprecipiation": 20,
}

# =========================================================
# 4. 周目录
# =========================================================
all_week_dirs = [
    d for d in root_dir.iterdir()
    if d.is_dir() and re.match(r"^\d+\s+\d{8}$", d.name)
]

all_week_dirs = sorted(all_week_dirs, key=lambda x: int(x.name.split()[0]))

current_index = next(
    (i for i, d in enumerate(all_week_dirs) if d.name == current_folder),
    None
)

if current_index is None:
    raise RuntimeError("未找到当前目录")

# =========================================================
# 5. 工具函数
# =========================================================
def get_variable_type(filename):
    for k in merge_rules:
        if k in filename:
            return k
    return None


def get_file_prefix(filename):
    return re.sub(r"-\d{8}\.nc$", "", filename)


# =========================================================
# 6. ⭐ 时间标准化：统一 valid_time
# =========================================================
def standardize_time(ds):

    if "valid_time" in ds:
        ds["valid_time"] = xr.decode_cf(ds).get("valid_time", ds["valid_time"])
        ds = ds.sortby("valid_time")

    elif "time" in ds:
        ds = ds.rename({"time": "valid_time"})
        ds["valid_time"] = ds["valid_time"].values
        ds = ds.sortby("valid_time")

    else:
        raise ValueError("找不到 time / valid_time")

    return ds


# =========================================================
# 7. ⭐ 核心合并（完全对齐第一段逻辑）
# =========================================================
def merge_nc_files(nc_files, output_file):

    ds_old = None
    ds_merge = None

    for i, f in enumerate(nc_files):

        print(f"读取: {f}")
        ds_new = xr.open_dataset(f)
        ds_new = standardize_time(ds_new)

        # =====================================================
        # 第一个文件作为基础
        # =====================================================
        if ds_old is None:
            ds_old = ds_new
            continue

        # =====================================================
        # overlap 检查（与第一段代码一致）
        # =====================================================
        time_old = ds_old.valid_time.values
        time_new = ds_new.valid_time.values

        overlap = np.intersect1d(time_old, time_new)

        if overlap.size > 0:
            print(f"   ⚠ 发现 {overlap.size} 个重叠时间，自动剔除")
            ds_new = ds_new.sel(valid_time=~ds_new.valid_time.isin(overlap))
        else:
            print("   ✔ 无时间重叠")

        # =====================================================
        # 如果没有新增时间
        # =====================================================
        if ds_new.valid_time.size == 0:
            print("   ⏭ 无新增时间，跳过该文件")
            ds_new.close()
            continue

        # =====================================================
        # concat + 排序（逐步累积）
        # =====================================================
        ds_old = xr.concat(
            [ds_old, ds_new],
            dim="valid_time"
        ).sortby("valid_time")

        # =====================================================
        # 双保险去重检查
        # =====================================================
        t = ds_old.valid_time.values
        if len(t) != len(np.unique(t)):
            raise RuntimeError("❌ 合并后仍存在重复时间！")

        ds_new.close()

    ds_merge = ds_old

    # =========================================================
    # 保存
    # =========================================================
    ds_merge.to_netcdf(output_file)

    ds_old.close()
    ds_merge.close()


# =========================================================
# 8. 文件列表
# =========================================================
download_files = sorted(
    f for f in os.listdir(dir_download) if f.endswith(".nc")
)

print("当前文件：")
for f in download_files:
    print(f)

# =========================================================
# 9. 主循环
# =========================================================
for current_file in download_files:

    print("\n=================================================")
    print(f"处理: {current_file}")

    variable_type = get_variable_type(current_file)

    if variable_type is None:
        print("跳过")
        continue

    need_num = merge_rules[variable_type]
    current_prefix = get_file_prefix(current_file)

    start_index = max(0, current_index - need_num + 1)
    selected_dirs = all_week_dirs[start_index: current_index + 1]

    nc_files = []

    for d in selected_dirs:

        download_dir = d / "数据" / "download"

        if not download_dir.exists():
            continue

        all_nc = sorted(f for f in os.listdir(download_dir) if f.endswith(".nc"))

        for f in all_nc:
            if get_file_prefix(f) == current_prefix:
                nc_files.append(str(download_dir / f))
                break

    print(f"找到文件数: {len(nc_files)}")

    if len(nc_files) == 0:
        print("跳过")
        continue

    output_file = dir_out / current_file

    print("开始合并...")

    merge_nc_files(nc_files, output_file)

    print(f"完成: {output_file}")

print("\n🎉 全部完成")

当前文件：
ERA5-daily-200hpa-Geopotential-20260831.nc
ERA5-daily-300hpa-Geopotential-20260831.nc
ERA5-daily-500hpa-Geopotential-20260831.nc
ERA5-daily-700hPa-SpecificHumidity-20260831.nc
ERA5-daily-800hPa-CloudFraction-20260831.nc
ERA5-daily-850hpa-Geopotential-20260831.nc
ERA5-daily-900hPa-Divergence-20260831.nc
ERA5-daily-900hPa-PotentialVorticity-20260831.nc
ERA5-daily-single level-2m_Temperature-20260831.nc
ERA5-daily-single level-MSLP-20260831.nc
ERA5-daily-single level-Totalprecipiation-20260831.nc
ERA5-monthly-single level-AZN-20260831.nc
ERA5-monthly-single level-tp-20260831.nc

处理: ERA5-daily-200hpa-Geopotential-20260831.nc
找到文件数: 20
开始合并...
读取: /Users/zhangnan/日常文件/ai/每周下载和预测/34 20260420/数据/download/ERA5-daily-200hpa-Geopotential-20260420.nc
读取: /Users/zhangnan/日常文件/ai/每周下载和预测/35 20260427/数据/download/ERA5-daily-200hpa-Geopotential-20260427.nc
   ✔ 无时间重叠
读取: /Users/zhangnan/日常文件/ai/每周下载和预测/36 20260504/数据/download/ERA5-daily-200hpa-Geopotential-20260504.nc
   ✔ 无时间重叠
读取: /Users/zhan

In [2]:
import xarray as xr

# 文件路径
file_path = "/Users/zhangnan/日常文件/ai/每周下载和预测/22 20260126/数据/download/ERA5-daily-200hpa-Geopotential-20260126.nc"

# 读取数据
ds = xr.open_dataset(file_path)

# 打印数据结构（建议先看一下）
print(ds)

# =========================================================
# 1. 查看所有坐标（确认时间变量名）
# =========================================================
print("\nCoordinates:")
print(ds.coords)

print("\nVariables:")
print(ds.data_vars)

# =========================================================
# 2. 自动识别时间变量（常见：time / valid_time）
# =========================================================
if "time" in ds.coords:
    time_var = ds["time"]
elif "valid_time" in ds.coords:
    time_var = ds["valid_time"]
else:
    raise ValueError("未找到 time 或 valid_time 变量")

# =========================================================
# 3. 输出时间信息
# =========================================================
print("\n时间维度信息：")
print(time_var)

# =========================================================
# 4. 转换为可读 datetime
# =========================================================
print("\n时间列表（前10个）：")
print(time_var.values[:20])

# =========================================================
# 5. 如果需要转 pandas datetime
# =========================================================
import pandas as pd

time_pd = pd.to_datetime(time_var.values)

print("\n转换为 pandas datetime（前10个）：")
print(time_pd[:20])

<xarray.Dataset> Size: 17MB
Dimensions:         (valid_time: 4, pressure_level: 1, latitude: 721,
                     longitude: 1440)
Coordinates:
  * valid_time      (valid_time) datetime64[ns] 32B 2026-01-01 ... 2026-01-04
  * pressure_level  (pressure_level) float64 8B 200.0
  * latitude        (latitude) float64 6kB 90.0 89.75 89.5 ... -89.75 -90.0
  * longitude       (longitude) float64 12kB 0.0 0.25 0.5 ... 359.2 359.5 359.8
    number          int64 8B ...
Data variables:
    z               (valid_time, pressure_level, latitude, longitude) float32 17MB ...
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2026-01-10T15:07 GRIB to CDM+CF via cfgrib-0.9.1...

Coordinates:
Coordinates:
  * valid_time      (valid_time) datetime64[ns